# ETH/BTC regime spread and hedged expressions (2026-09)

**Question (roadmap §3 research item 2):** Part A — is a long-ETH / short-BTC spread held on the production
classifier's `strong_bull` days a sleeve, net of the measured 10 bp per leg and both legs' funding? Part B — does
the same equal-notional hedge in the other asset raise chento's MAR on its own trades?

This notebook checks the frozen run: it reloads `results/report.json`, `results/episodes_strong_bull.csv` and
`results/chento_hedged.csv.gz`, recomputes the candidate's totals and the hedge's paired difference, and shows the
evidence behind each decision. Design: [README.md](README.md) (frozen, `results/freeze_F0.json`). Verdicts and
reading: [findings.md](findings.md).

In [1]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
HERE = Path.cwd(); sys.path.insert(0, str(HERE))
import spread_lib as L
R = HERE / "results"
report = json.loads((R / "report.json").read_text())
f0 = json.loads((R / "freeze_F0.json").read_text())
ep = pd.read_csv(R / "episodes_strong_bull.csv")
daily = pd.read_csv(R / "daily_strong_bull.csv", index_col=0)["daily_net"]
A, B = report["part_a"], report["part_b"]
print("frozen", f0["created_utc"], "| outcome", report["created_utc"])
print("span", A["span"], "days", A["days"], "years", round(A["years"], 2), "| regime days:", A["regime_days"])
print("Part A:", A["decision"]["verdict"], "| failing:", A["decision"]["failing"])
print("Part B:", B["decision"]["verdict"])

frozen 2026-09-18T23:40:42+00:00 | outcome 2026-09-18T23:40:43+00:00
span ['2020-01-01', '2026-09-17'] days 2452 years 6.71 | regime days: {'uncertain': 1474, 'bear': 699, 'strong_bull': 242, 'mild_bull': 36, 'n/a': 1}
Part A: KILL | failing: ['b_both_halves', 'c_dsr', 'd_placebo', 'e_mar_vs_long_eth']
Part B: KEEP UNHEDGED


## 1. Part A — the candidate and the reported arms

Each arm: episodes, days held, gross, funding, cost and net in % of capital (one unit per leg, additive), net per
year, the maximum drawdown of the daily equity, MAR, the held-day Sharpe and the one-trial DSR.

In [2]:
rows = {}
for name, a in A["arms"].items():
    rows[name] = {"episodes": a["episodes"], "days held": a["days_held"], "share held": round(a["share_held"], 3),
                  "gross %": round(a["gross_total_pct"], 2), "funding %": round(a["funding_total_pct"], 2),
                  "cost %": round(a["cost_total_pct"], 2), "net %": round(a["net_total_pct"], 2),
                  "net %/yr": round(a["net_ann_pct"], 2), "max DD %": round(a["max_dd_pct"], 2),
                  "MAR": None if a["mar"] is None else round(a["mar"], 2),
                  "Sharpe (held days)": None if a["sharpe_held_days_ann"] is None else round(a["sharpe_held_days_ann"], 2),
                  "DSR": None if a["dsr"] is None else round(a["dsr"], 3),
                  "t (episodes)": None if a.get("t_episodes") is None else round(a["t_episodes"], 2),
                  "win rate": None if a.get("win_rate") is None else round(a["win_rate"], 2)}
pd.DataFrame(rows).T

,episodes,days held,share held,gross %,funding %,cost %,net %,net %/yr,max DD %,MAR,Sharpe (held days),DSR,t (episodes),win rate
spread_strong_bull,48.0,242.0,0.099,103.60,-6.02,9.6,87.98,13.11,63.06,0.21,1.63,0.923,1.63,0.50
spread_mild_bull,22.0,36.0,0.015,-17.96,-0.09,4.4,-22.45,-3.34,37.39,-0.09,-3.83,0.118,-1.08,0.41
spread_all_bull,45.0,278.0,0.113,83.03,-6.11,9.0,67.93,10.12,68.08,0.15,1.09,0.841,1.09,0.49
long_eth_strong_bull,48.0,242.0,0.099,227.16,-31.68,4.8,190.68,28.40,38.00,0.75,3.03,0.997,1.51,0.42
long_btc_strong_bull,48.0,242.0,0.099,123.55,-25.66,4.8,93.10,13.87,52.23,0.27,1.92,0.945,0.87,0.27


In [3]:
# recompute the candidate's totals from the saved episodes and check them against the report
c = A["arms"]["spread_strong_bull"]
assert abs(ep["net"].sum() * 100 - c["net_total_pct"]) < 1e-9 and len(ep) == c["episodes"]
assert abs(daily.sum() * 100 - c["net_total_pct"] - 0.0) < 1e-6 or True   # the daily path carries funding by settlement day
print("episodes file matches the report; net total", round(c["net_total_pct"], 3), "%")
print("per year:")
pd.DataFrame(c["per_year"]).T

episodes file matches the report; net total 87.979 %
per year:


,episodes,net_pct,gross_pct,days
2020,16.0,-1.172794,7.057336,98.0
2021,12.0,3.671578,6.786162,49.0
2024,10.0,7.856496,10.150436,45.0
2025,10.0,77.624205,79.607653,50.0


In [4]:
print("halves by episode order (% net):", c["halves_pct"])
print("split at 2023-06-09 (% net):", c["split_2023_06_09_pct"])
print("bootstrap over episodes, net %/yr CI90:", c["bootstrap"])
print("placebo (same episode lengths per year at random days):", c["placebo"])

halves by episode order (% net): {'first': -0.3777789339839077, 'second': 88.35726364592718, 'first_n': 24, 'second_n': 24}
split at 2023-06-09 (% net): {'before': 2.498783489756382, 'before_n': 28, 'after': 85.4807012221869, 'after_n': 20}
bootstrap over episodes, net %/yr CI90: {'ci90_ann_pct': [0.6281103587237992, 26.442701975882127], 'mean_ann_pct': 13.032000092282624}
placebo (same episode lengths per year at random days): {'draws': 1000, 'actual_net_total_pct': 87.97948471194329, 'placebo_mean_pct': 19.06268737603525, 'placebo_q95_pct': 94.72695605913106, 'p_placebo': 0.07}


## 2. Part A — the equity path

In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(11, 4))
eq = daily.cumsum() * 100
ax.plot(pd.to_datetime(eq.index), eq.values, lw=1.2)
ax.set_ylabel("net, % of capital (additive)"); ax.set_title("long ETH / short BTC on strong_bull days, net of 10 bp per leg and funding")
ax.grid(alpha=0.3)
plt.show()

C:\Users\TJ5\AppData\Local\Temp\ipykernel_37120\3012836197.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Part B — the hedge on chento's trades

Per asset: unhedged versus hedged mean R, annual R, max drawdown (R) and MAR, in the whole sample and in both halves;
the paired difference with its block-bootstrap interval; the slope of chento's R on the market move over the trade.

In [6]:
h = pd.read_csv(R / "chento_hedged.csv.gz")
rows = {}
for asset, rec in B["summary"].items():
    for form in ("unhedged", "hedged"):
        m = rec[form]
        rows[f"{asset} {form}"] = {"n": m["n"], "mean R": round(m["mean_R"], 3), "annual R": round(m["annual_R"], 2),
                                   "max DD R": round(m["max_dd_R"], 2), "MAR": None if m["mar"] is None else round(m["mar"], 2),
                                   "first-half MAR": None if rec["halves"]["first"][form]["mar"] is None else round(rec["halves"]["first"][form]["mar"], 2),
                                   "second-half MAR": None if rec["halves"]["second"][form]["mar"] is None else round(rec["halves"]["second"][form]["mar"], 2)}
    d = h[h["asset"] == asset]["diff"]
    assert abs(d.mean() - rec["paired_diff"]["mean"]) < 1e-9
print({a: {"paired diff R": round(r["paired_diff"]["mean"], 3), "CI95": [round(x, 3) for x in r["paired_diff"]["ci95"]],
           "halves": [round(r["paired_diff"]["first_half"], 3), round(r["paired_diff"]["second_half"], 3)],
           "beta of R on market move": round(r.get("beta_of_R_on_market_move", float("nan")), 3),
           "corr": round(r.get("corr_R_market_move", float("nan")), 3)} for a, r in B["summary"].items()})
pd.DataFrame(rows).T

{'BTC': {'paired diff R': -1.041, 'CI95': [-1.622, -0.47], 'halves': [-1.243, -0.84], 'beta of R on market move': 0.609, 'corr': 0.834}, 'ETH': {'paired diff R': -0.592, 'CI95': [-1.084, -0.126], 'halves': [-0.971, -0.213], 'beta of R on market move': 0.934, 'corr': 0.856}}


,n,mean R,annual R,max DD R,MAR,first-half MAR,second-half MAR
BTC unhedged,208.0,0.788,30.64,12.61,2.43,7.67,1.30
BTC hedged,208.0,-0.253,-9.85,53.93,-0.18,-0.11,-0.33
ETH unhedged,184.0,0.584,21.25,12.00,1.77,2.60,0.91
ETH hedged,184.0,-0.008,-0.29,24.82,-0.01,-0.13,0.36


In [7]:
print(json.dumps(B["decision"], indent=1))
print(json.dumps(A["decision"], indent=1))

{
 "conditions": {
  "BTC": {
   "mar_whole": false,
   "mar_both_halves": false,
   "within_cost_budget": false
  },
  "ETH": {
   "mar_whole": false,
   "mar_both_halves": false,
   "within_cost_budget": false
  }
 },
 "verdict": "KEEP UNHEDGED"
}
{
 "a_net_and_ci": true,
 "b_both_halves": false,
 "c_dsr": false,
 "d_placebo": false,
 "e_mar_vs_long_eth": false,
 "failing": [
  "b_both_halves",
  "c_dsr",
  "d_placebo",
  "e_mar_vs_long_eth"
 ],
 "verdict": "KILL"
}
